# Урок 8 · Челленджи

Три мини-проекта посложнее ДЗ, в каждом по 3 шага: **1** — учёт расходов, **2** — дни рождения группы, **3** — жеребьёвка команд. В каждом пригодятся функции, модули и обработка ошибок сразу. Шаги идут по порядку, и каждый использует предыдущий.

**Как работать.** Сначала запустите ячейку с функциями `check` и `check_raises`. В каждом проекте запустите ячейку с данными, потом решайте шаги по порядку: замените `...` на свой код и запустите — ✅ или ❌ покажут результат. Если застряли — вернитесь к тетради урока 8 и ДЗ.

In [1]:
# ▶ Запустите эту ячейку первой — она нужна для проверки задач
def check(name, func, expected):
    try:
        got = func()
    except Exception as e:
        print(f"❌ {name}: ошибка {type(e).__name__}: {e}")
        return
    if got == expected:
        print(f"✅ {name}")
    else:
        print(f"❌ {name}: получилось {got!r}, а нужно {expected!r}")

def check_raises(name, func, error_type):
    try:
        func()
    except error_type:
        print(f"✅ {name}")
        return
    except Exception as e:
        print(f"❌ {name}: ошибка {type(e).__name__}, а нужна {error_type.__name__}")
        return
    print(f"❌ {name}: ошибки не было, а нужна {error_type.__name__}")

print("Готово!")

Готово!


---
# Челлендж 1. Учёт расходов

Вы весь месяц записывали расходы в заметки строками вида `дата;категория;сумма`. Иногда записывали с ошибками. Нужно разобрать записи, отбросить неправильные и посчитать траты.

In [2]:
from datetime import date, datetime

lines = [
    "01.11.2026;еда;3500",
    "02.11.2026;транспорт;250",
    "02.11.2026;еда;-100",
    "03.11.2026;кино;2500",
    "вчера;еда;1200",
    "04.11.2026;транспорт;250",
    "05.11.2026;еда;много",
    "05.11.2026;еда",
    "06.11.2026;еда;4100",
]

### Шаг 1. Разобрать одну запись
`parse_expense(line)` возвращает кортеж `(дата, категория, сумма)`, где дата — объект `date`, а сумма — целое число. Если запись неправильная, функция бросает `ValueError`:
- не три части через `;` — `raise ValueError("Нужно 3 поля: дата;категория;сумма")`;
- дата не в формате `дд.мм.гггг` или сумма не число — эти ошибки `strptime` и `int` бросят сами;
- сумма меньше или равна нулю — `raise ValueError("Сумма должна быть больше нуля")`.

In [7]:
def parse_expense(line):
    l = line.split(';')
    if len(l) != 3:
        raise ValueError("Нужно 3 поля: дата;категория;сумма")
    date_str, category, amount = l
    try:
        dt = datetime.strptime(date_str, "%d.%m.%Y").date()
    except ValueError:
        raise ValueError("Неверный формат даты")
    try:
        amount_n = int(amount)
    except ValueError:
        raise ValueError("Сумма не число")
    else:
        if amount_n <= 0:
            raise ValueError("Сумма должна быть больше нуля")
    return (dt, category, amount_n)

check("правильная", lambda: parse_expense("01.11.2026;еда;3500"), (date(2026, 11, 1), "еда", 3500))
check_raises("минус", lambda: parse_expense("02.11.2026;еда;-100"), ValueError)
check_raises("дата словом", lambda: parse_expense("вчера;еда;1200"), ValueError)
check_raises("сумма словом", lambda: parse_expense("05.11.2026;еда;много"), ValueError)
check_raises("не хватает поля", lambda: parse_expense("05.11.2026;еда"), ValueError)

✅ правильная
✅ минус
✅ дата словом
✅ сумма словом
✅ не хватает поля


### Шаг 2. Разобрать все записи
`load_expenses(lines)` возвращает кортеж `(правильные_записи, сколько_ошибок)`. Каждую строку разбирайте через `parse_expense`; если она бросила `ValueError`, считайте ошибку и идите дальше.

In [17]:
def load_expenses(lines):
    rcd = []
    cnt = 0
    for l in lines:
        try:
            tp = parse_expense(l)
        except ValueError:
            cnt += 1
        else:
            rcd.append(l)
    return (rcd, cnt)


records, bad = load_expenses(lines)
print("Правильных:", len(records), "| с ошибками:", bad)
check("правильных", lambda: len(records), 5)
check("с ошибками", lambda: bad, 4)

Правильных: 5 | с ошибками: 4
✅ правильных
✅ с ошибками


### Шаг 3. Сколько потрачено
`total(records, *categories)` — сумма трат по указанным категориям. Если категории не переданы — сумма всех трат.

Подсказка: отберите нужные записи через `filter` и `lambda`, затем сложите суммы (третий элемент кортежа).

In [33]:
def total(records, *categories):
    if categories:
        return sum([int(i.split(';')[2]) if i.split(';')[1] in categories else 0 for i in records])
    else:
        return sum([int(i.split(';')[2]) for i in records])

check("всё", lambda: total(records), 10600)
check("еда", lambda: total(records, "еда"), 7600)
check("транспорт и кино", lambda: total(records, "транспорт", "кино"), 3000)
check("нет такой категории", lambda: total(records, "одежда"), 0)

✅ всё
✅ еда
✅ транспорт и кино
✅ нет такой категории


---
# Челлендж 2. Дни рождения группы

Куратор собрал даты рождения студентов, но в двух записях ошибки. Посчитаем возраст, найдём неправильные записи и узнаем, чей день рождения ближе всего.

Чтобы результаты у всех совпадали, «сегодня» зафиксировано: `TODAY = date(2026, 11, 5)`.

In [34]:
from datetime import date, datetime

TODAY = date(2026, 11, 5)

group = {
    "Алия":   "15.03.2007",
    "Тимур":  "02.11.2005",
    "Асель":  "31.12.2006",
    "Ерлан":  "20.07.2030",
    "Данияр": "30.02.2006",
}

### Шаг 1. Возраст
`age_on(birthday_text, today)` возвращает полный возраст в годах.
- Неправильную дату (например, `30.02`) `strptime` отклонит сам — бросит `ValueError`.
- Если дата рождения позже `today` — `raise ValueError("Дата рождения в будущем")`.

Как считать возраст: `today.year - год_рождения`, и ещё минус 1, если день рождения в этом году ещё не наступил. Проверка «ещё не наступил»: `(today.month, today.day) < (b.month, b.day)` — кортежи сравниваются по очереди: сначала месяц, при равенстве день.

In [35]:
def age_on(birthday_text, today):
    bd_date = datetime.strptime(birthday_text, "%d.%m.%Y").date()
    if bd_date > today:
        raise ValueError("Дата рождения в будущем")
    if (today.month, today.day) < (bd_date.month, bd_date.day):
        return today.year - bd_date.year - 1
    else:
        return today.year - bd_date.year


check("Алия", lambda: age_on("15.03.2007", TODAY), 19)
check("Тимур (уже был)", lambda: age_on("02.11.2005", TODAY), 21)
check("Асель (ещё не было)", lambda: age_on("31.12.2006", TODAY), 19)
check_raises("из будущего", lambda: age_on("20.07.2030", TODAY), ValueError)
check_raises("30 февраля", lambda: age_on("30.02.2006", TODAY), ValueError)

✅ Алия
✅ Тимур (уже был)
✅ Асель (ещё не было)
✅ из будущего
✅ 30 февраля


### Шаг 2. Вся группа
`group_ages(group, today)` возвращает кортеж из двух частей:
1. словарь «имя → возраст» для правильных записей;
2. **отсортированный список** имён, у кого в дате ошибка.

In [41]:
def group_ages(group, today):
    mst = []
    crd = {}
    for i in group:
        try:
            cr = age_on(group[i], today)
        except ValueError:
            mst.append(i)
        else:
            crd.update({i: cr})
    
    return (crd, sorted(mst))


ages, errors = group_ages(group, TODAY)
print(ages, errors)
check("возрасты", lambda: ages, {"Алия": 19, "Тимур": 21, "Асель": 19})
check("ошибки", lambda: errors, ["Данияр", "Ерлан"])

{'Алия': 19, 'Тимур': 21, 'Асель': 19} ['Данияр', 'Ерлан']
✅ возрасты
✅ ошибки


### Шаг 3. Чей день рождения ближе
1. `days_to_birthday(birthday_text, today)` — через сколько дней следующий день рождения. Соберите дату дня рождения в году `today.year`; если она уже прошла (меньше `today`), возьмите следующий год. Если день рождения сегодня — `0`.
2. `upcoming` — имена из `ages` (только правильные записи), отсортированные по тому, чей день рождения ближе. Используйте `sorted` с `key=lambda`.

Подсказка: `date(год, b.month, b.day)` собирает дату с нужным годом.

In [44]:
def days_to_birthday(birthday_text, today):
    return (datetime.strptime(birthday_text, "%d.%m.%Y").date().replace(year=today.year+1) - today).days % 365

upcoming = sorted([i for i in ages], key=lambda x: days_to_birthday(group[x], TODAY))

print(upcoming)
check("Асель", lambda: days_to_birthday("31.12.2006", TODAY), 56)
check("Алия", lambda: days_to_birthday("15.03.2007", TODAY), 130)
check("Тимур (уже прошёл)", lambda: days_to_birthday("02.11.2005", TODAY), 362)
check("сегодня", lambda: days_to_birthday("05.11.2000", TODAY), 0)
check("порядок", lambda: upcoming, ["Асель", "Алия", "Тимур"])

['Асель', 'Алия', 'Тимур']
✅ Асель
✅ Алия
✅ Тимур (уже прошёл)
✅ сегодня
✅ порядок


---
# Челлендж 3. Жеребьёвка команд

Группу нужно случайно разделить на команды для проекта, посчитать баллы команд и красиво объявить результат.

In [45]:
import random

students = ["Алия", "Тимур", "Асель", "Ерлан", "Данияр", "Мадина", "Айгерим", "Нурлан"]

### Шаг 1. Разделить на команды
`make_teams(names, n_teams, seed=None)` возвращает список команд (каждая команда — список имён).
- Если `n_teams < 1` или команд больше, чем людей — `raise ValueError("Неверное число команд")`.
- Исходный список **не должен меняться**: перемешивайте копию `names[:]`.
- Перед перемешиванием вызовите `random.seed(seed)` — тогда с одинаковым `seed` жеребьёвка повторяется.
- Раздавайте людей по очереди: первый — в команду 0, второй — в 1, ... Номер команды для `i`-го человека — `i % n_teams`.

In [67]:
def make_teams(names, n_teams, seed=None):
    if n_teams < 1 or n_teams > len(names):
        raise ValueError("Неверное число команд")
    n_names = names[:]
    random.seed(seed)
    random.shuffle(n_names)

    cm = [[] for _ in range(n_teams)]
    for i in range(len(names)):
        cm[i % n_teams].append(n_names[i])
        # print(cm)
    return cm

teams = make_teams(students, 3, seed=1)
print(teams)
check("размеры команд", lambda: sorted(map(len, teams), reverse=True), [3, 3, 2])
check("все на месте", lambda: sorted(sum(teams, [])), sorted(students))
check("seed повторяет", lambda: make_teams(students, 3, seed=7) == make_teams(students, 3, seed=7), True)
check("список не изменился", lambda: students[0], "Алия")
check_raises("0 команд", lambda: make_teams(students, 0), ValueError)
check_raises("команд больше людей", lambda: make_teams(["Алия", "Тимур"], 5), ValueError)

[['Ерлан', 'Мадина', 'Данияр'], ['Айгерим', 'Нурлан', 'Асель'], ['Тимур', 'Алия']]
✅ размеры команд
✅ все на месте
✅ seed повторяет
✅ список не изменился
✅ 0 команд
✅ команд больше людей


### Шаг 2. Баллы команд
`team_scores(teams, scores)` получает команды и словарь «имя → баллы» и возвращает список сумм баллов каждой команды. Если у кого-то нет баллов в словаре, поймайте `KeyError` и бросьте понятную ошибку: `raise ValueError(f"Нет баллов для: {name}")`.

In [74]:
def team_scores(teams, scores):
    team_score = []
    for team in teams:
        cnt_score = 0
        for name in team:
            try:
                cnt_score += scores[name]
            except KeyError:
                raise ValueError(f"Нет баллов для {name}")
        team_score.append(cnt_score)
    return team_score
        

scores = {"Алия": 9, "Тимур": 6, "Асель": 8, "Ерлан": 5}
check("суммы", lambda: team_scores([["Алия", "Тимур"], ["Асель", "Ерлан"]], scores), [15, 13])
check_raises("нет баллов", lambda: team_scores([["Алия", "Мадина"]], scores), ValueError)

✅ суммы
✅ нет баллов


### Шаг 3. Объявление
`announce(teams, totals, **options)` возвращает текст объявления — строки, склеенные через `"\n"`. Каждая строка: `Команда 1: Алия, Тимур`. Настройки через `**options`:
- `title="..."` — если передан, первой строкой идёт заголовок;
- `show_scores=True` — в конце каждой строки дописать ` — 15` (баллы команды).

Подсказка: `options.get("title")`, `options.get("show_scores", False)` и `", ".join(team)`.

In [83]:
def announce(teams, totals, **options):
    txt = ""
    if options.get("title", ''):
        txt += options.get("title") + "\n"
    for i in range(len(teams)):
        txt += f"Команда {i+1}: " + ", ".join(teams[i])
        if options.get("show_scores", False):
            txt += f" — {totals[i]}"
        txt += "\n"
    return txt.strip("\n")

t = [["Алия", "Тимур"], ["Асель", "Ерлан"]]
print(announce(t, [15, 13], title="Итоги проекта", show_scores=True))
check("просто", lambda: announce(t, [15, 13]), "Команда 1: Алия, Тимур\nКоманда 2: Асель, Ерлан")
check("с баллами", lambda: announce(t, [15, 13], show_scores=True), "Команда 1: Алия, Тимур — 15\nКоманда 2: Асель, Ерлан — 13")
check("с заголовком", lambda: announce(t, [15, 13], title="Финал").split("\n")[0], "Финал")

Итоги проекта
Команда 1: Алия, Тимур — 15
Команда 2: Асель, Ерлан — 13
✅ просто
✅ с баллами
✅ с заголовком


In [12]:
# x = "dfg, dfd!"
# # print(x.strip(",!"))



dfg, dfd
